In [ ]:
from pathlib import Path

import anndata as ad
import seaborn as sns
import squidpy as sq
from spatialleiden import distance2connectivity, spatialleiden

from batch_integration.multisample_utils import (
    preprocess_genes,
    preprocess_neighbors,
    preprocess_scaling,
)
from utils import DATA_DIR, RESULT_DIR, remove_tick_and_label

In [ ]:
# Settings
seed = 42

In [ ]:
file = DATA_DIR / "3D_lymphnode" / "GSE251926_metastatic_lymph_node_3d.h5ad"

results_lymphnode = RESULT_DIR / "3D_lymphnode"
results_lymphnode.mkdir(exist_ok=True, parents=True)

# Preprocess

In [ ]:
adata = ad.read_h5ad(file)
adata.X = adata.layers["raw"].astype(int)

del adata.layers

In [ ]:
adata.obs.annotation.cat.categories

In [ ]:
sq.gr.spatial_neighbors(
    adata, coord_type="generic", n_neighs=10, spatial_key="spatial_3d_aligned"
)

In [ ]:
_ = sns.histplot(adata.obsp["spatial_distances"].data, binrange=(0, 120), bins=120)

In [ ]:
preprocess_genes(adata, batch_key="n_section")
preprocess_scaling(adata, seed=42)

In [ ]:
preprocess_neighbors(adata, "X_pca")

# Clustering

In [ ]:
adata.obsp["spatial_connectivities"] = distance2connectivity(
    adata.obsp["spatial_distances"]
)

In [ ]:
spatialleiden(adata, directed=(False, True), layer_ratio=1.8, seed=seed, n_iterations=3)

In [ ]:
columns = ["spatialleiden", "n_section"]

labels = adata.obs[columns]
labels.loc[:, ["x", "y", "z"]] = adata.obsm["spatial_3d_aligned"]
labels.to_parquet(results_lymphnode / "no_batch_integration.parquet")

# Plotting

In [ ]:
scatter_kwargs = {
    "s": 1,
    "linewidth": 0,
}

In [ ]:
import pandas as pd

labels = pd.read_parquet(results_lymphnode / "no_batch_integration.parquet").astype(
    {"spatialleiden": "category"}
)

In [ ]:
import colorcet as cc

cluster = labels["spatialleiden"].cat.categories

palette = dict(zip(cluster, sns.color_palette(cc.glasbey, n_colors=len(cluster))))

g = sns.FacetGrid(labels, col="n_section", col_wrap=3, aspect=1.2)
_ = g.map_dataframe(
    sns.scatterplot,
    x="x",
    y="y",
    hue="spatialleiden",
    palette=palette,
    **scatter_kwargs,
)
for ax in g.axes.flat:
    ax.set(aspect=1)
    remove_tick_and_label(ax)

_ = g.tight_layout()
g.savefig(results_lymphnode / "no_batch_integration.png")